<center>
    <h1>Probabilistic models taken by Storm</h1>
    <h2>Step 3: Working with policies</h2>
    <br>
    <br>
    <img src="images/storm_logo.png" width="500px"/>
    <h3>Matthias Volk</h3>
</center>

<div align="right">Press <em>spacebar</em> to navigate</div>


# Preparation
Load models from before

In [ ]:
# Import prism model from the previous step
import stormvogel
import stormpy

from examples.orchard_builder import build_simple, build_prism

In [ ]:
orchard_simple = build_simple()
# We use the prism encoding in this part of the tutorial.
orchard_prism, prism_program = build_prism()
# We are primarily interested in the probability of winning.
formula = stormpy.parse_properties('Pmax=? [F "PlayersWon"]')[0]

In [ ]:
# Helper function
def model_check(model, prop):
    formula = stormpy.parse_properties(prop)[0]
    result = stormpy.model_checking(model, formula, only_initial_states=True)
    return result.at(model.initial_states[0])

In [ ]:
# Skip

## Winning probability
- *What is the maximal probability that the players win the game?*

In [ ]:
prob_players_won = 'Pmax=? [F "PlayersWon"]'
result = stormvogel.model_checking(orchard_simple, prob_players_won)
print(result.get_result_of_state(orchard_simple.get_initial_state()))

- **What is this optimal winning strategy?**

## Computing policies
- need to explicitly enable policy extraction due to extra bookkeeping

In [ ]:
result = stormpy.model_checking(orchard_prism, formula, extract_scheduler=True)

- policy is memoryless<br>
  $\to$ print optimal action per state

In [ ]:
string_scheduler = str(result.scheduler)
import os
max_lines = 20
print(os.linesep.join(string_scheduler.split(os.linesep)[:max_lines]))

In [ ]:
# Skip

## Computing policies
- improvement: display state information (internal variables)
- further improvement: only display states with actual choices

In [ ]:
i = 0
for state in orchard_prism.states:
    choice = result.scheduler.get_choice(state)
    action_index = choice.get_deterministic_choice()
    action = state.actions[action_index]
    action_name = next(iter(action.labels))
    if True: # action_name.startswith("choose"):
        print(f"In state {state.valuations} choose action {action_name}")
        i+=1
        if i>=max_lines:
            break

In [ ]:
# Skip

## Induced submodel
- applying the policy $\sigma$ yields a DTMC $\mathcal{M}^\sigma$
- allows further analysis w.r.t. this policy

In [ ]:
# Get induced model
induced = orchard_prism.apply_scheduler(result.scheduler, True)
# Analysis
all_cherries = 'Pmax=? [F "AllCherriesPicked"]'
print(f"Prob for fixed: {model_check(induced, all_cherries)}")
print(f"Prob for optimal: {model_check(orchard_prism, all_cherries)}")

In [ ]:
# Skip

# Compact policies
- Problem: policy represented as state-action pairs:<br>
  $\to$ understanding/explaining the optimal policy nearly impossible for larger models
- Need a compact representation of policies

### Approaches for compact policies
- decision tree learning in [dtControl tool](https://dtcontrol.model.in.tum.de/):
  - heuristically create small trees for a policy
- dtMap in [PAYNT tool](https://github.com/randriu/synthesis):
  - find a minimal decision tree for a given policy.

Both interface with Storm

# Compact policies via dtMAP
- [PAYNT tool](https://github.com/randriu/synthesis)
- introduce additional predicate `most_F`
  - indicates that fruit type `F` has most fruits left
  - otherwise not compactly representatable as a decision tree over linear predicates

### Preparation

In [ ]:
def _declare_extra_orchard_variables(manager):
    apple_variable : stormpy.Variable = manager.get_variable("apple")
    plum_variable : stormpy.Variable = manager.get_variable("plum")
    pear_variable : stormpy.Variable = manager.get_variable("pear")
    cherry_variable : stormpy.Variable = manager.get_variable("cherry")
    # The else cases below are only relevant for rerunning this function multiple times.
    if not manager.has_variable("most_apples"):
        most_apples_variable = manager.create_boolean_variable("most_apples")
    else:
        most_apples_variable = manager.get_variable("most_apples")
    if not manager.has_variable("most_pears"):
        most_pears_variable = manager.create_boolean_variable("most_pears")
    else:
        most_pears_variable = manager.get_variable("most_pears")
    if not manager.has_variable("most_plums"):
        most_plums_variable = manager.create_boolean_variable("most_plums")
    else:
        most_plums_variable = manager.get_variable("most_plums")
    if not manager.has_variable("most_cherries"):
        most_cherries_variable = manager.create_boolean_variable("most_cherries")
    else:
        most_cherries_variable = manager.get_variable("most_cherries")
    
    apple_variable_expression = apple_variable.get_expression()
    plum_variable_expression = plum_variable.get_expression()
    pear_variable_expression = pear_variable.get_expression()
    cherry_variable_expression = cherry_variable.get_expression()
    # Add the extra definitions
    extra_definitions = []
    extra_definitions.append((most_apples_variable, stormpy.Expression.Conjunction(
        [stormpy.Expression.Geq(apple_variable_expression, plum_variable_expression),
         stormpy.Expression.Geq(apple_variable_expression, cherry_variable_expression),
         stormpy.Expression.Geq(apple_variable_expression, pear_variable_expression)])))
    extra_definitions.append((most_pears_variable, stormpy.Expression.Conjunction(
        [stormpy.Expression.Geq(pear_variable_expression, plum_variable_expression),
         stormpy.Expression.Geq(pear_variable_expression, cherry_variable_expression),
         stormpy.Expression.Geq(pear_variable_expression, apple_variable_expression)])))
    extra_definitions.append((most_plums_variable, stormpy.Expression.Conjunction(
        [stormpy.Expression.Geq(plum_variable_expression, pear_variable_expression),
         stormpy.Expression.Geq(plum_variable_expression, cherry_variable_expression),
         stormpy.Expression.Geq(plum_variable_expression, apple_variable_expression)])))
    extra_definitions.append((most_cherries_variable, stormpy.Expression.Conjunction(
        [stormpy.Expression.Geq(cherry_variable_expression, pear_variable_expression),
         stormpy.Expression.Geq(cherry_variable_expression, plum_variable_expression),
         stormpy.Expression.Geq(cherry_variable_expression, apple_variable_expression)])))
    return extra_definitions

In [ ]:
# We allow a few options here.
add_additional_definitions = True
"""add_additional_definitions adds expressions saying that one fruit is among the types of fruit that we have most."""
maintain_old_valuations = False
"""maintain_old_valuations preserves the old variables and their assignments"""
copy_fruit_amounts = False
"""copy_fruit_amounts preserves the amount of fruit of every type, useful if we are not maintaining the old valuations."""
assert not maintain_old_valuations or not copy_fruit_amounts

# Create the transformer.
svt = stormpy.StateValuationTransformer(orchard_prism.state_valuations)
# Add extra Boolean definitions:
if add_additional_definitions:
    extra_definitions = _declare_extra_orchard_variables(prism_program.expression_manager)
    for (v, definition) in extra_definitions:
        svt.add_boolean_expression(v, definition)
# Add integer definitions. Here, we consider copies of the number of items, only useful we do not maintain the old variables.
if copy_fruit_amounts:
    manager = prism_program.expression_manager
    svt.add_integer_expression(manager.create_integer_variable("nr_apples"), manager.get_variable("apple").get_expression())
    svt.add_integer_expression(manager.create_integer_variable("nr_plums"), manager.get_variable("plum").get_expression())
    svt.add_integer_expression(manager.create_integer_variable("nr_cherries"), manager.get_variable("cherry").get_expression())
    svt.add_integer_expression(manager.create_integer_variable("nr_pears"), manager.get_variable("pear").get_expression())
# Create a new MDP.
new_mdp = stormpy.set_state_valuations(orchard_prism, svt.build(maintain_old_valuations))

## Run synthesis

In [ ]:
import paynt

In [ ]:
# We run model checking once more, and optionally extract a scheduler.
mc_result = stormpy.model_checking(new_mdp, formula, extract_scheduler=False)
print(f"Overall optimal result: {mc_result.at(new_mdp.initial_states[0])}")

In [ ]:
colored_mdp_factory = paynt.dt.DtColoredMdpFactory(new_mdp)
task = paynt.dt.DtTask([formula], 3)
result = paynt.dt.synthesize(colored_mdp_factory, task)
print("success:", result.success)

In [ ]:
# Skip

## Compact policy: visualization

In [ ]:
import io
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Convert to graphviz format
png_bytes = result.tree.to_graphviz().pipe(format='png')
# Plot
img = mpimg.imread(io.BytesIO(png_bytes))
fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(img)
ax.axis('off')
fig.tight_layout()
plt.show(fig)

In [ ]:
# Skip

![Decision tree](images/dt.png)